# ⚡ LLM Quantization — A Practical Deep Dive

> **Quantization** is the process of reducing the numerical precision of a model's weights (and optionally activations) from high-precision formats (e.g., FP32 or FP16) to lower-precision formats (e.g., INT8 or INT4). For Large Language Models (LLMs), this dramatically reduces memory footprint and increases inference throughput — often with negligible accuracy loss.

---

## 🧭 Why Quantize LLMs?

| Problem | Quantization Solution |
|---|---|
| LLMs are too large to fit on consumer GPUs | 4-bit quant shrinks a 7B model from ~14 GB → ~4 GB |
| Slow inference due to memory bandwidth bottlenecks | Smaller weights → faster memory transfers |
| High cloud inference costs | Smaller model → fewer GPUs required |
| Edge / on-device deployment | INT4/INT8 runs on CPUs and mobile chips |

---

## 📋 Notebook Sections

| # | Section | Topics Covered |
|---|---|---|
| 1 | **Fundamentals of Quantization** | Data types, scale/zero-point, symmetric vs. asymmetric |
| 2 | **Quantization from Scratch (NumPy)** | Manual INT8 quantization to build intuition |
| 3 | **Post-Training Quantization (PTQ)** | Absmax, zero-point quantization, rounding error analysis |
| 4 | **LLM INT8 via `bitsandbytes`** | Load a real LLM in INT8 with `load_in_8bit=True` |
| 5 | **LLM INT4 via `bitsandbytes` (NF4)** | QLoRA-style 4-bit with double quantization |
| 6 | **GPTQ Quantization** | Weight-only post-training quantization |
| 7 | **AWQ Quantization** | Activation-aware weight quantization |
| 8 | **GGUF / llama.cpp Format** | CPU-friendly quantization for local inference |
| 9 | **Benchmarking & Comparison** | Memory, speed, and perplexity trade-offs |
| 10 | **Choosing the Right Method** | Decision guide |

---

## 🔑 Key Concepts at a Glance

- **Bit-width:** Number of bits used to represent a value. FP32 = 32 bits, FP16 = 16 bits, INT8 = 8 bits, INT4 = 4 bits.
- **PTQ (Post-Training Quantization):** Quantize a pre-trained model without retraining.
- **QAT (Quantization-Aware Training):** Simulate quantization during training so the model adapts.
- **Weight-only quantization:** Only weights are quantized; activations remain in FP16 during compute.
- **NF4 (NormalFloat4):** A 4-bit data type designed specifically for normally distributed neural network weights.

---
## 1. Imports & Environment Setup

We install and import all required libraries:

- **`numpy`** — For the from-scratch quantization examples.
- **`torch`** — PyTorch for model operations.
- **`transformers`** — Hugging Face library to load pre-trained LLMs.
- **`bitsandbytes`** — Enables INT8 and INT4 (NF4) quantization via `BitsAndBytesConfig`.
- **`matplotlib`** — Visualisations.

> **Installation:** Run the cell below the first time to install missing packages. Comment it out afterwards.

In [ ]:
# Uncomment and run once to install dependencies
# !pip install -q transformers accelerate bitsandbytes sentencepiece optimum auto-gptq autoawq

In [ ]:
# ── Standard libraries ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── PyTorch ────────────────────────────────────────────────────────────────────
import torch

# ── Hugging Face ───────────────────────────────────────────────────────────────
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

# ── Utility ────────────────────────────────────────────────────────────────────
import gc      # Garbage collection to free GPU memory between experiments
import time

# ── Device Configuration ───────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device          : {DEVICE}")
print(f"PyTorch version : {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {total_mem:.1f} GB")

---
## 2. Fundamentals of Quantization

### 2.1 Numeric Data Types Used in Deep Learning

| Format | Bits | Range | Precision | Typical Use |
|---|---|---|---|---|
| FP32 | 32 | ±3.4 × 10³⁸ | ~7 decimal digits | Training (full precision) |
| FP16 | 16 | ±65,504 | ~3 decimal digits | Mixed-precision training / inference |
| BF16 | 16 | ±3.4 × 10³⁸ | ~2 decimal digits | Training on TPUs/Ampere GPUs |
| INT8 | 8 | [-128, 127] | Integer only | Inference (8-bit quantization) |
| INT4 | 4 | [-8, 7] | Integer only | Aggressive inference compression |
| NF4 | 4 | [-1, 1] (non-linear) | Normal-dist optimised | QLoRA 4-bit fine-tuning |

### 2.2 The Quantization Formula

The general linear quantization mapping is:

$$x_{\text{quant}} = \text{clamp}\left(\text{round}\left(\frac{x}{s}\right) + z,\; q_{\min},\; q_{\max}\right)$$

And the **de-quantization** (reconstruction) formula:

$$x_{\text{recon}} = s \cdot (x_{\text{quant}} - z)$$

Where:
- $s$ = **scale factor** — maps the floating-point range to the integer range.
- $z$ = **zero-point** — integer offset so that 0.0 (FP) maps to exactly $z$ (INT).
- $q_{\min}, q_{\max}$ = minimum and maximum representable integer values (e.g., -128, 127 for INT8).

### 2.3 Symmetric vs. Asymmetric Quantization

| Type | Zero-point | Formula | Use Case |
|---|---|---|---|
| **Symmetric** | `z = 0` | $s = \max(|x|) / (2^{b-1} - 1)$ | Weights (usually symmetric around 0) |
| **Asymmetric** | `z ≠ 0` | $s = (x_{\max} - x_{\min}) / (2^b - 1)$ | Activations (often non-negative after ReLU) |

In [ ]:
# ── Visualise Memory Savings by Bit-Width ──────────────────────────────────────

# A 7B parameter model — memory in GB for each dtype
model_params = 7e9  # 7 billion parameters
dtype_info = {
    'FP32\n(32-bit)': {'bits': 32, 'color': '#e74c3c'},
    'FP16/BF16\n(16-bit)': {'bits': 16, 'color': '#e67e22'},
    'INT8\n(8-bit)': {'bits': 8, 'color': '#2ecc71'},
    'INT4\n(4-bit)': {'bits': 4, 'color': '#3498db'},
    'INT2\n(2-bit)': {'bits': 2, 'color': '#9b59b6'},
}

labels   = list(dtype_info.keys())
memories = [model_params * info['bits'] / 8 / 1e9 for info in dtype_info.values()]
colors   = [info['color'] for info in dtype_info.values()]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels, memories, color=colors, edgecolor='black', linewidth=0.6)

for bar, mem in zip(bars, memories):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f'{mem:.1f} GB',
        va='center', fontsize=11, fontweight='bold'
    )

ax.set_xlabel('Memory Required (GB)', fontsize=12)
ax.set_title('Memory Footprint of a 7B Parameter Model by Precision', fontsize=13, pad=10)
ax.axvline(x=24, color='gray', linestyle='--', linewidth=1.2, label='24 GB VRAM (e.g., RTX 3090)')
ax.axvline(x=8, color='black', linestyle=':', linewidth=1.2, label='8 GB VRAM (e.g., RTX 3070)')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print("\nMemory Summary for a 7B Model:")
for label, mem in zip(labels, memories):
    label_clean = label.replace('\n', ' ')
    print(f"  {label_clean:<20}: {mem:.2f} GB")

---
## 3. Quantization from Scratch with NumPy

Before using library tools, let's build intuition by implementing both **symmetric** and **asymmetric** INT8 quantization manually.

This helps understand:
- How scale and zero-point are computed.
- Where **quantization error** (rounding loss) comes from.
- The trade-off between quantization range coverage and precision.

In [ ]:
def symmetric_quantize(x: np.ndarray, num_bits: int = 8):
    """
    Symmetric INT quantization: zero-point is fixed at 0.
    Maps [-abs_max, +abs_max] → [-q_max, +q_max]

    Args:
        x:        Input floating-point array.
        num_bits: Target bit-width (e.g., 8 for INT8).

    Returns:
        x_quant:  Quantized integer array.
        scale:    Scale factor used for de-quantization.
    """
    q_max = 2 ** (num_bits - 1) - 1          # E.g., 127 for INT8
    abs_max = np.max(np.abs(x))              # Use absolute max for symmetry

    scale = abs_max / q_max                  # Maps float range → int range

    # Quantize: divide by scale, round to nearest integer, clamp to valid range
    x_quant = np.clip(np.round(x / scale), -q_max, q_max).astype(np.int8)

    return x_quant, scale


def symmetric_dequantize(x_quant: np.ndarray, scale: float) -> np.ndarray:
    """Reconstruct float values from symmetric INT quantization."""
    return x_quant.astype(np.float32) * scale   # Multiply back by scale


# ── Demo ───────────────────────────────────────────────────────────────────────
np.random.seed(42)
# Simulate a typical weight tensor: normally distributed around 0
weights_fp32 = np.random.randn(5).astype(np.float32) * 0.5

weights_int8, scale = symmetric_quantize(weights_fp32, num_bits=8)
weights_reconstructed = symmetric_dequantize(weights_int8, scale)

print("=" * 55)
print("  Symmetric INT8 Quantization Demo")
print("=" * 55)
print(f"Scale factor : {scale:.6f}")
print()
print(f"{'Original FP32':>20} | {'Quantized INT8':>14} | {'Reconstructed':>14} | {'Error':>10}")
print("-" * 65)
for orig, q, rec in zip(weights_fp32, weights_int8, weights_reconstructed):
    error = abs(orig - rec)
    print(f"{orig:>20.6f} | {int(q):>14d} | {rec:>14.6f} | {error:>10.6f}")

mse = np.mean((weights_fp32 - weights_reconstructed) ** 2)
print(f"\nMean Squared Error (quantization noise): {mse:.8f}")

In [ ]:
def asymmetric_quantize(x: np.ndarray, num_bits: int = 8):
    """
    Asymmetric INT quantization: uses a non-zero zero-point.
    Better for asymmetric distributions (e.g., ReLU activations: all values >= 0).

    Args:
        x:        Input floating-point array.
        num_bits: Target bit-width.

    Returns:
        x_quant:     Quantized unsigned integer array.
        scale:       Scale factor.
        zero_point:  Integer offset to align 0.0 in float space.
    """
    q_min = 0
    q_max = 2 ** num_bits - 1                              # E.g., 255 for UINT8

    x_min, x_max = x.min(), x.max()

    # Scale maps the full float range to the full int range
    scale = (x_max - x_min) / (q_max - q_min)

    # Zero-point ensures 0.0 (float) maps to an integer
    zero_point = int(np.round(q_min - x_min / scale))
    zero_point = int(np.clip(zero_point, q_min, q_max))

    # Quantize
    x_quant = np.clip(np.round(x / scale) + zero_point, q_min, q_max).astype(np.uint8)

    return x_quant, scale, zero_point


def asymmetric_dequantize(x_quant: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """Reconstruct float values from asymmetric quantization."""
    return scale * (x_quant.astype(np.float32) - zero_point)


# ── Demo on ReLU-style activations (non-negative) ──────────────────────────────
activations_fp32 = np.abs(np.random.randn(5).astype(np.float32))   # All >= 0 (post-ReLU)

act_int8, scale, zp = asymmetric_quantize(activations_fp32, num_bits=8)
act_reconstructed    = asymmetric_dequantize(act_int8, scale, zp)

print("=" * 60)
print("  Asymmetric UINT8 Quantization Demo (ReLU activations)")
print("=" * 60)
print(f"Scale      : {scale:.6f}")
print(f"Zero-point : {zp}")
print()
print(f"{'Original FP32':>20} | {'Quantized UINT8':>15} | {'Reconstructed':>14} | {'Error':>10}")
print("-" * 65)
for orig, q, rec in zip(activations_fp32, act_int8, act_reconstructed):
    error = abs(orig - rec)
    print(f"{orig:>20.6f} | {int(q):>15d} | {rec:>14.6f} | {error:>10.6f}")

mse = np.mean((activations_fp32 - act_reconstructed) ** 2)
print(f"\nMean Squared Error: {mse:.8f}")

### 3.1 Quantization Error vs. Bit-Width

As we reduce bit-width, fewer discrete levels are available, so more information is lost in rounding. The plot below shows how Mean Squared Error (MSE) grows as bit-width decreases.

In [ ]:
# Simulate a large weight tensor (e.g., one transformer attention matrix)
np.random.seed(0)
weight_tensor = np.random.randn(1024).astype(np.float32) * 0.02  # Small std, typical for LLMs

bit_widths = [8, 6, 5, 4, 3, 2]
mse_values = []

for bits in bit_widths:
    q, s = symmetric_quantize(weight_tensor, num_bits=bits)
    recon = symmetric_dequantize(q, s)
    mse   = np.mean((weight_tensor - recon) ** 2)
    mse_values.append(mse)
    print(f"  {bits}-bit : MSE = {mse:.2e}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: MSE vs. bit-width
axes[0].plot(bit_widths, mse_values, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[0].set_xlabel('Bit-width', fontsize=12)
axes[0].set_ylabel('Mean Squared Error (Quantization Noise)', fontsize=11)
axes[0].set_title('Quantization Error vs. Bit-Width', fontsize=13)
axes[0].invert_xaxis()           # Lower bits on the right (decreasing precision)
axes[0].set_yscale('log')        # Log scale to see the exponential growth
axes[0].grid(True, alpha=0.4)
axes[0].spines[['top', 'right']].set_visible(False)

# Annotate key points
for bw, mse in zip(bit_widths, mse_values):
    axes[0].annotate(f'{bw}-bit', xy=(bw, mse), xytext=(5, 5), textcoords='offset points', fontsize=9)

# Right: Original vs. 4-bit reconstructed weight distribution
q4, s4 = symmetric_quantize(weight_tensor, num_bits=4)
recon4 = symmetric_dequantize(q4, s4)

axes[1].hist(weight_tensor, bins=60, alpha=0.6, label='Original FP32', color='#3498db')
axes[1].hist(recon4, bins=60, alpha=0.6, label='Reconstructed INT4', color='#e74c3c')
axes[1].set_xlabel('Weight Value', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Weight Distribution: FP32 vs. INT4 Reconstruction', fontsize=12)
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

---
## 4. INT8 Quantization with `bitsandbytes` (LLM.int8())

**`bitsandbytes`** provides the `LLM.int8()` method by Tim Dettmers et al. (2022). It enables loading models in **8-bit precision** with almost no accuracy degradation.

### How LLM.int8() Works

Naïve INT8 quantization fails for LLMs because of **outlier features** — a small fraction (~0.1%) of activation values that are orders of magnitude larger than the rest, causing massive quantization error.

LLM.int8() solves this with **mixed-precision decomposition**:
1. **Detect outlier columns** (values exceeding a threshold, typically `|x| > 6`).
2. **Compute outlier matrix-multiplications in FP16** (high precision for important features).
3. **Compute the remaining 99.9% in INT8** (fast and memory-efficient).
4. Merge results in FP16.

**Result:** ~2× memory savings with <1% accuracy drop on most models.

```
FP32 model (7B) : ~28 GB
FP16 model (7B) : ~14 GB
INT8 model (7B) : ~7 GB   ← ~2× saving vs FP16
```

In [ ]:
# ── INT8 Quantization Configuration ───────────────────────────────────────────

bnb_int8_config = BitsAndBytesConfig(
    load_in_8bit=True,              # Enable LLM.int8() quantization
    llm_int8_threshold=6.0,         # Outlier threshold: values above this use FP16
    llm_int8_skip_modules=None,     # List module names to skip (e.g., ['lm_head'])
    llm_int8_enable_fp32_cpu_offload=False,  # Set True to offload some layers to CPU
)

print("INT8 BitsAndBytesConfig:")
print(f"  load_in_8bit     : {bnb_int8_config.load_in_8bit}")
print(f"  INT8 threshold   : {bnb_int8_config.llm_int8_threshold}")
print()
print("Usage example:")
print("  model = AutoModelForCausalLM.from_pretrained(")
print("      model_id,")
print("      quantization_config=bnb_int8_config,")
print("      device_map='auto'")
print("  )")

In [ ]:
# ── Load a Model in INT8 ───────────────────────────────────────────────────────
# We use 'facebook/opt-125m' — a small 125M parameter model for quick demos.
# Replace with 'meta-llama/Llama-2-7b-hf' etc. on a capable GPU.

MODEL_ID = "facebook/opt-125m"    # ~250MB download; swap for a larger model as needed

print(f"Loading '{MODEL_ID}' in INT8...")

tokenizer_int8 = AutoTokenizer.from_pretrained(MODEL_ID)

# BitsAndBytesConfig requires a CUDA GPU; fall back to FP16 on CPU-only systems
if DEVICE == "cuda":
    model_int8 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_int8_config,
        device_map="auto",         # Automatically place layers on GPU(s)/CPU
    )
    print("INT8 model loaded on GPU ✅")
else:
    # CPU fallback — load in FP32 for compatibility
    model_int8 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    print("⚠️  No CUDA detected — loaded in FP32 (INT8 requires GPU). INT8 demo skipped.")

print(f"\nModel class : {type(model_int8).__name__}")

In [ ]:
def get_model_memory_mb(model) -> float:
    """
    Compute the total memory occupied by model parameters in MB.

    Args:
        model: A PyTorch nn.Module.

    Returns:
        Memory in megabytes.
    """
    total_bytes = sum(
        p.numel() * p.element_size()    # element_size() = bytes per element
        for p in model.parameters()
    )
    return total_bytes / 1e6  # Convert bytes to MB


memory_int8 = get_model_memory_mb(model_int8)
print(f"Model memory (INT8) : {memory_int8:.1f} MB")

# Count parameters
total_params = sum(p.numel() for p in model_int8.parameters())
print(f"Total parameters    : {total_params / 1e6:.1f} M")

In [ ]:
# ── Inference with the INT8 Model ─────────────────────────────────────────────
prompt = "The key advantage of model quantization in LLMs is"

inputs = tokenizer_int8(prompt, return_tensors="pt")

# Move inputs to the same device as the model
if DEVICE == "cuda":
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

print(f"Prompt: {prompt!r}\n")

with torch.no_grad():
    start = time.time()
    output_ids = model_int8.generate(
        **inputs,
        max_new_tokens=60,          # Generate up to 60 new tokens
        do_sample=False,            # Greedy decoding for deterministic output
        temperature=1.0,
    )
    elapsed = time.time() - start

generated_text = tokenizer_int8.decode(output_ids[0], skip_special_tokens=True)
new_tokens     = output_ids.shape[1] - inputs['input_ids'].shape[1]

print(f"Generated (INT8 model):\n{generated_text}")
print(f"\n⏱  Time: {elapsed:.2f}s | Tokens generated: {new_tokens} | Tokens/sec: {new_tokens / elapsed:.1f}")

---
## 5. INT4 Quantization with NF4 (QLoRA / `bitsandbytes`)

**NF4 (NormalFloat4)** is a 4-bit data type introduced in the **QLoRA** paper (Dettmers et al., 2023). It is specifically designed for quantizing normally distributed weight tensors, which are ubiquitous in neural networks.

### Why NF4 is Better Than Naive INT4

Standard INT4 quantizes into 16 uniformly spaced levels. But neural network weights follow a **normal distribution** — values near 0 are far more common than large values. NF4 uses **non-uniform quantization levels** that are optimally spaced according to the quantiles of a standard normal distribution, minimising distortion.

### Double Quantization

QLoRA further introduces **double quantization**: the scale factors themselves are quantized (from FP32 to FP8), saving an additional ~0.5 bits per parameter.

```
Memory savings (7B model):
  FP16  → ~14 GB
  INT8  → ~7  GB  (2× saving)
  NF4   → ~3.5 GB (4× saving)
  NF4 + DQ → ~3.0 GB
```

### Compute Data Type

During forward pass computation, weights are **de-quantized back to BF16** on-the-fly for the matrix multiplication. This means the model computes in BF16 precision but stores weights in NF4, striking a balance between accuracy and memory.

In [ ]:
# ── NF4 / INT4 BitsAndBytesConfig ─────────────────────────────────────────────

bnb_nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,                           # Enable 4-bit NF4 quantization
    bnb_4bit_quant_type="nf4",                   # 'nf4' (NormalFloat4) or 'fp4' (Float4)
    bnb_4bit_compute_dtype=torch.bfloat16,       # Compute dtype during forward pass
    bnb_4bit_use_double_quant=True,              # Double quantization saves ~0.5 bits/param
)

print("NF4 BitsAndBytesConfig (QLoRA-style):")
print(f"  load_in_4bit              : {bnb_nf4_config.load_in_4bit}")
print(f"  bnb_4bit_quant_type       : {bnb_nf4_config.bnb_4bit_quant_type}")
print(f"  bnb_4bit_compute_dtype    : {bnb_nf4_config.bnb_4bit_compute_dtype}")
print(f"  bnb_4bit_use_double_quant : {bnb_nf4_config.bnb_4bit_use_double_quant}")

In [ ]:
# ── Load Model in NF4 ─────────────────────────────────────────────────────────
print(f"Loading '{MODEL_ID}' in NF4 (4-bit)...")

# Free INT8 model from memory before loading another
del model_int8
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

if DEVICE == "cuda":
    model_nf4 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_nf4_config,
        device_map="auto",
    )
    print("NF4 model loaded on GPU ✅")
else:
    model_nf4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    print("⚠️  No CUDA — loaded in FP32 (NF4 requires GPU). NF4 demo skipped.")

memory_nf4 = get_model_memory_mb(model_nf4)
print(f"\nModel memory (NF4)  : {memory_nf4:.1f} MB")

In [ ]:
# ── Inference with the NF4 Model ──────────────────────────────────────────────
tokenizer_nf4 = AutoTokenizer.from_pretrained(MODEL_ID)

inputs_nf4 = tokenizer_nf4(prompt, return_tensors="pt")
if DEVICE == "cuda":
    inputs_nf4 = {k: v.to(DEVICE) for k, v in inputs_nf4.items()}

with torch.no_grad():
    start = time.time()
    output_ids_nf4 = model_nf4.generate(
        **inputs_nf4,
        max_new_tokens=60,
        do_sample=False,
    )
    elapsed_nf4 = time.time() - start

generated_nf4  = tokenizer_nf4.decode(output_ids_nf4[0], skip_special_tokens=True)
new_tokens_nf4 = output_ids_nf4.shape[1] - inputs_nf4['input_ids'].shape[1]

print(f"Generated (NF4 model):\n{generated_nf4}")
print(f"\n⏱  Time: {elapsed_nf4:.2f}s | Tokens generated: {new_tokens_nf4} | Tokens/sec: {new_tokens_nf4 / elapsed_nf4:.1f}")

---
## 6. GPTQ — Post-Training Quantization for Transformers

**GPTQ** (Frantar et al., 2022) is a **weight-only** post-training quantization method based on the Optimal Brain Compression (OBC) framework. It achieves INT4 (and even INT3/INT2) with excellent quality by:

1. Processing the model **layer by layer**, one weight matrix at a time.
2. Using a **small calibration dataset** (e.g., 128 samples) to observe activation statistics.
3. **Minimising the layer-wise quantization error** by updating remaining (unquantized) weights to compensate for the error introduced by already-quantized weights.

### GPTQ vs. bitsandbytes INT4

| Aspect | GPTQ | bitsandbytes NF4 |
|---|---|---|
| Approach | Error-compensating PTQ | Absmax quantization |
| Quality | Generally better at INT4 | Simpler but fast |
| Speed | Slow to quantize (hours) | Instant at load time |
| Pre-quantized models | ✅ Available on Hugging Face | Quantized on-the-fly |
| CPU inference | Via llama.cpp / GGUF | No |

### Loading a Pre-Quantized GPTQ Model

Many popular models (Llama, Mistral, Falcon, etc.) have pre-quantized GPTQ versions on Hugging Face, making this approach practical without running the quantization yourself.

In [ ]:
# ── GPTQ Configuration & Loading Example ──────────────────────────────────────
# Note: 'auto-gptq' or 'optimum' must be installed.
#       Pre-quantized GPTQ models are available on Hugging Face:
#       e.g., 'TheBloke/Llama-2-7B-GPTQ'

# Conceptual loading code (works when 'optimum' is installed):
GPTQ_MODEL_ID = "TheBloke/opt-125m-GPTQ"   # Example GPTQ model

print("GPTQ Loading Pattern:")
print("""
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load a pre-quantized GPTQ model — no need to run quantization yourself!
model = AutoModelForCausalLM.from_pretrained(
    "TheBloke/Llama-2-7B-GPTQ",   # Pre-quantized to INT4 (GPTQ)
    device_map="auto",
    trust_remote_code=False,
    revision="main",               # Specify the GPTQ revision/branch
)
tokenizer = AutoTokenizer.from_pretrained("TheBloke/Llama-2-7B-GPTQ")
""")

print("\nGPTQ Quantization Parameters (used during offline quantization):")
gptq_params = {
    'bits'            : '4 (INT4) — or 3, 8',
    'group_size'      : '128 (quantization granularity per group of weights)',
    'desc_act'        : 'True (use activation order for better accuracy)',
    'damp_percent'    : '0.01 (dampening factor for Hessian inversion stability)',
    'dataset'         : '"c4" or "wikitext2" (calibration dataset, ~128 samples)',
}
for k, v in gptq_params.items():
    print(f"  {k:<20}: {v}")

---
## 7. AWQ — Activation-Aware Weight Quantization

**AWQ** (Lin et al., 2023) is a recent INT4 weight quantization method that achieves better quality than GPTQ by protecting **salient weights** — those that have the most impact on output quality.

### The Core Insight

Not all weights are equally important. AWQ observes that ~1% of weights correspond to large activations and dominate model performance. Instead of quantizing them separately (like LLM.int8() does for activations), AWQ scales the **input channels** to equalise weight magnitudes before quantization:

$$W' = W / s, \quad X' = X \times s$$

where $s$ is a per-channel scale chosen so that large activations are balanced out, making all weights easier to quantize uniformly.

### AWQ vs GPTQ

| Aspect | AWQ | GPTQ |
|---|---|---|
| Core idea | Scale salient channels | OBC error compensation |
| Perplexity | ✅ Often slightly better | Competitive |
| Speed (quantization) | Faster than GPTQ | Slow (hours) |
| Hardware efficiency | Better for GPU kernels | Comparable |
| Pre-quantized models | ✅ Available (e.g., TheBloke) | ✅ Available |

In [ ]:
# ── AWQ Loading Pattern ────────────────────────────────────────────────────────
print("AWQ Loading Pattern (requires 'autoawq'):")
print("""
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# Load pre-quantized AWQ model from Hugging Face
model = AutoAWQForCausalLM.from_quantized(
    "TheBloke/Llama-2-7B-AWQ",
    fuse_layers=True,      # Fuse layers for faster inference
    trust_remote_code=False,
    safetensors=True,
)
tokenizer = AutoTokenizer.from_pretrained("TheBloke/Llama-2-7B-AWQ")
""")

print("\nAWQ Quantization Config (for offline quantization):")
awq_config = {
    'zero_point'  : 'True — use asymmetric quantization',
    'q_group_size': '128 — group size for weight quantization',
    'w_bit'       : '4 — target bit-width',
    'version'     : 'GEMM — kernel type (GEMM or GEMV)',
}
for k, v in awq_config.items():
    print(f"  {k:<15}: {v}")

---
## 8. GGUF Format & llama.cpp

**GGUF** (GPT-Generated Unified Format) is a binary file format developed for the **llama.cpp** project. It enables running quantized LLMs efficiently on CPUs (and Apple Silicon MPS).

### Key Features of GGUF

- **CPU-first:** Optimised for systems without a dedicated GPU.
- **Multiple quantization levels in one file:** A GGUF file may contain metadata about the quantization level (Q2_K, Q4_K_M, Q5_K_S, Q8_0, etc.).
- **Mmap support:** Large models can be memory-mapped — only the parts needed for each forward pass are loaded into RAM.
- **Cross-platform:** Runs on macOS (Metal), Windows, Linux, ARM.

### GGUF Quantization Levels

| Format | Bits/Weight | Notes | Quality |
|---|---|---|---|
| `Q2_K` | ~2.6 | Very aggressive, noticeable quality drop | ⭐⭐ |
| `Q3_K_M` | ~3.4 | Medium 3-bit | ⭐⭐⭐ |
| `Q4_0` | ~4.0 | Baseline 4-bit | ⭐⭐⭐ |
| `Q4_K_M` | ~4.5 | **Recommended**: Best quality/size trade-off | ⭐⭐⭐⭐ |
| `Q5_K_M` | ~5.7 | High quality, larger size | ⭐⭐⭐⭐ |
| `Q6_K` | ~6.6 | Near-lossless | ⭐⭐⭐⭐⭐ |
| `Q8_0` | ~8.0 | Almost identical to FP16 | ⭐⭐⭐⭐⭐ |
| `F16` | 16.0 | Original half-precision | ⭐⭐⭐⭐⭐ |

> **`Q4_K_M` is the community standard** for local LLM inference — it gives ~4.5 bits per weight with excellent quality retention.

In [ ]:
# ── GGUF with llama.cpp / llama-cpp-python ────────────────────────────────────
# This code shows the typical usage pattern.
# Install: pip install llama-cpp-python

print("GGUF / llama-cpp-python usage pattern:")
print("""
from llama_cpp import Llama

# Load a GGUF model (downloaded from Hugging Face)
# Example: TheBloke/Llama-2-7B-Chat-GGUF → llama-2-7b-chat.Q4_K_M.gguf
model = Llama(
    model_path="./llama-2-7b-chat.Q4_K_M.gguf",
    n_ctx=4096,           # Context window length
    n_threads=8,          # CPU threads for inference
    n_gpu_layers=0,       # Set > 0 to offload layers to GPU (requires CUDA build)
    verbose=False,
)

# Generate text
output = model(
    "Explain quantization in simple terms:",
    max_tokens=200,
    temperature=0.7,
    stop=["\\n\\n"],
)
print(output['choices'][0]['text'])
""")

# ── Visualise GGUF Quantization Quality vs Size ────────────────────────────────
gguf_levels = {
    'Q2_K':   {'size_gb': 2.8,  'ppl_delta': 0.65},   # ppl_delta = perplexity increase vs FP16
    'Q3_K_M': {'size_gb': 3.3,  'ppl_delta': 0.25},
    'Q4_0':   {'size_gb': 3.9,  'ppl_delta': 0.15},
    'Q4_K_M': {'size_gb': 4.1,  'ppl_delta': 0.10},
    'Q5_K_M': {'size_gb': 4.8,  'ppl_delta': 0.04},
    'Q6_K':   {'size_gb': 5.5,  'ppl_delta': 0.01},
    'Q8_0':   {'size_gb': 7.2,  'ppl_delta': 0.003},
    'F16':    {'size_gb': 13.5, 'ppl_delta': 0.0},
}

names  = list(gguf_levels.keys())
sizes  = [v['size_gb'] for v in gguf_levels.values()]
deltas = [v['ppl_delta'] for v in gguf_levels.values()]

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()   # Secondary y-axis for perplexity delta

x = np.arange(len(names))
bars = ax1.bar(x, sizes, color='#3498db', alpha=0.7, label='Model Size (GB)', width=0.4)
ax2.plot(x, deltas, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Perplexity Δ vs F16')

# Highlight the recommended level
recommended_idx = names.index('Q4_K_M')
bars[recommended_idx].set_color('#2ecc71')
bars[recommended_idx].set_edgecolor('black')
bars[recommended_idx].set_linewidth(1.5)

ax1.set_xticks(x)
ax1.set_xticklabels(names, fontsize=10)
ax1.set_ylabel('Model Size (GB)', fontsize=11, color='#3498db')
ax2.set_ylabel('Perplexity Δ (vs F16, lower = better)', fontsize=11, color='#e74c3c')
ax1.set_title('GGUF Quantization Levels — Size vs. Quality (7B model)', fontsize=13, pad=10)

green_patch = mpatches.Patch(color='#2ecc71', label='Recommended: Q4_K_M')
blue_patch  = mpatches.Patch(color='#3498db', alpha=0.7, label='Model Size (GB)')
red_line    = plt.Line2D([0], [0], color='#e74c3c', linewidth=2, marker='o', label='Perplexity Δ')
ax1.legend(handles=[green_patch, blue_patch, red_line], loc='upper left', fontsize=9)

ax1.spines[['top']].set_visible(False)
plt.tight_layout()
plt.show()

---
## 9. Benchmarking & Comparison

We now compare the quantization methods across the three most important dimensions:

1. **Memory** — How much GPU/CPU RAM is required?
2. **Speed** — How many tokens per second can be generated?
3. **Quality** — How does perplexity (lower = better) compare to FP16?

The numbers below are representative benchmarks for a **7B parameter model** (e.g., Llama-2-7B) on an A100-80GB GPU.

In [ ]:
# ── Benchmark Data (representative figures for a 7B model on A100 80GB) ────────
# Sources: Papers + community benchmarks from EleutherAI, Hugging Face, TheBloke

benchmarks = {
    'Method':        ['FP32',  'FP16',  'INT8\n(bnb)',   'NF4\n(bnb)',  'GPTQ\nINT4', 'AWQ\nINT4', 'GGUF\nQ4_K_M'],
    'Memory (GB)':   [28.0,    14.0,    7.0,              3.5,           3.8,           3.6,          3.9],
    'Tokens/sec':    [18,      35,      28,               40,            55,            60,           25],    # CPU lower
    'Perplexity':    [5.47,    5.47,    5.52,             5.56,          5.59,          5.55,         5.60],  # WikiText-2
    'Ppl Δ vs FP16': [0.00,    0.00,    0.05,             0.09,          0.12,          0.08,         0.13],
}

methods  = benchmarks['Method']
memories = benchmarks['Memory (GB)']
speeds   = benchmarks['Tokens/sec']
ppl_base = benchmarks['Perplexity']
ppl_delta= benchmarks['Ppl Δ vs FP16']

# ── 3-Panel Comparison Plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c', '#f39c12']

# Panel 1: Memory
axes[0].bar(methods, memories, color=colors, edgecolor='black', linewidth=0.6)
axes[0].set_title('Memory Footprint', fontsize=13, fontweight='bold')
axes[0].set_ylabel('GPU Memory (GB)', fontsize=11)
axes[0].axhline(y=24, color='gray', linestyle='--', linewidth=1, label='24 GB limit')
axes[0].axhline(y=8,  color='black', linestyle=':', linewidth=1, label='8 GB limit')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', labelsize=8)
for bar, mem in zip(axes[0].patches, memories):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{mem:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Panel 2: Speed
axes[1].bar(methods, speeds, color=colors, edgecolor='black', linewidth=0.6)
axes[1].set_title('Inference Speed', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Tokens per Second', fontsize=11)
axes[1].tick_params(axis='x', labelsize=8)
for bar, spd in zip(axes[1].patches, speeds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{spd}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Panel 3: Perplexity (quality)
axes[2].bar(methods, ppl_base, color=colors, edgecolor='black', linewidth=0.6)
axes[2].set_title('Perplexity (WikiText-2)\nLower = Better Quality', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Perplexity', fontsize=11)
axes[2].set_ylim(5.35, 5.75)
axes[2].tick_params(axis='x', labelsize=8)
for bar, ppl in zip(axes[2].patches, ppl_base):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{ppl:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('LLM Quantization Methods — Benchmark Comparison (7B model, A100 80GB)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Print Benchmark Summary Table ─────────────────────────────────────────────
methods_clean = [m.replace('\n', ' ') for m in methods]

print("\n" + "=" * 75)
print(f"{'Method':<20} {'Memory (GB)':>12} {'Tokens/sec':>12} {'Perplexity':>12} {'PPL Δ vs FP16':>15}")
print("-" * 75)
for m, mem, spd, ppl, delta in zip(methods_clean, memories, speeds, ppl_base, ppl_delta):
    flag = "  ← baseline" if m in ('FP32', 'FP16') else ""
    print(f"{m:<20} {mem:>12.1f} {spd:>12d} {ppl:>12.2f} {delta:>15.2f}{flag}")
print("=" * 75)
print("\nNote: Benchmarks are approximate and hardware/model-dependent.")
print("      Perplexity values are illustrative for a 7B-class model.")

---
## 10. Choosing the Right Quantization Method

Use this decision guide to pick the right approach for your use case.

```
START
 │
 ├─ Do you have a GPU?
 │   ├─ NO  → GGUF (Q4_K_M) with llama.cpp for CPU inference
 │   └─ YES →
 │       ├─ VRAM ≥ 14 GB? → FP16 (best quality, no quantization needed)
 │       ├─ VRAM 8–14 GB? → INT8 (bitsandbytes) — simple, robust, 2× saving
 │       └─ VRAM < 8 GB?  →
 │           ├─ Best speed?    → AWQ (best GPU kernel efficiency)
 │           ├─ Best quality?  → GPTQ (OBC error compensation)
 │           └─ Fine-tuning?   → NF4 (bitsandbytes) + QLoRA
```

### Quick Reference

| Scenario | Recommended Method | Why |
|---|---|---|
| Fine-tuning on consumer GPU (≤12 GB) | **NF4 + QLoRA** | 4-bit storage + efficient adapter training |
| Production inference (GPU server) | **AWQ INT4** | Best speed + quality on GPU |
| Production inference (GPU server, safety-critical) | **INT8 (bnb)** | Minimal quality loss |
| Local/offline CPU inference | **GGUF Q4_K_M** | CPU-optimised, portable |
| Highest quality at 4-bit | **GPTQ INT4** | Error-compensating PTQ |
| MacBook / Apple Silicon | **GGUF + llama.cpp (Metal)** | Metal GPU acceleration |
| Edge / mobile deployment | **INT8 or INT4** via TFLite/CoreML | Platform-specific tooling |

In [ ]:
# ── Quality vs. Speed vs. Memory Scatter Plot ──────────────────────────────────
# Bubble size ∝ memory footprint

methods_clean = [m.replace('\n', ' ') for m in methods]

fig, ax = plt.subplots(figsize=(10, 6))

for i, (m, mem, spd, ppl) in enumerate(zip(methods_clean, memories, speeds, ppl_base)):
    ax.scatter(
        spd, ppl,
        s=mem * 30,             # Bubble area proportional to memory
        c=[colors[i]],
        alpha=0.8,
        edgecolors='black',
        linewidths=0.8,
        zorder=3,
        label=f"{m}  ({mem:.1f} GB)"
    )
    ax.annotate(
        m, xy=(spd, ppl),
        xytext=(5, 5), textcoords='offset points',
        fontsize=9, fontweight='bold'
    )

ax.set_xlabel('Inference Speed (tokens/sec)  →  Faster ⟶', fontsize=12)
ax.set_ylabel('Perplexity  →  ← Better', fontsize=12)
ax.invert_yaxis()           # Lower perplexity = better quality → top of chart
ax.set_title(
    'Speed vs. Quality for LLM Quantization Methods\n(Bubble size = memory footprint)',
    fontsize=13, pad=10
)
ax.legend(loc='lower right', fontsize=8, title='Method (Memory)')
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# Annotate the "sweet spot" region
ax.annotate(
    '"Sweet Spot"\nhigh speed + quality',
    xy=(58, 5.56), xytext=(35, 5.62),
    fontsize=9, color='gray',
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.2)
)

plt.tight_layout()
plt.show()

---
## 📊 Summary & Key Takeaways

### What We Covered

1. **Quantization Fundamentals** — Scale/zero-point, symmetric vs. asymmetric, bit-width trade-offs.
2. **Manual Implementation** — Built INT8 quantization from scratch to understand rounding error.
3. **`bitsandbytes` INT8** — LLM.int8() with mixed-precision outlier handling (~2× memory savings, <0.1% accuracy drop).
4. **`bitsandbytes` NF4 (4-bit)** — Normal-Float-4 with double quantization for QLoRA-style fine-tuning (~4× memory savings).
5. **GPTQ** — Error-compensating post-training quantization for maximum 4-bit quality.
6. **AWQ** — Activation-aware quantization, best GPU kernel efficiency at INT4.
7. **GGUF/llama.cpp** — Portable, CPU-first quantization for local deployment.
8. **Benchmarking** — Compared memory, speed, and perplexity across all methods.

### The Quantization Landscape (2024)

```
Highest Quality ──────────────────────────────────────────── Most Compressed

   FP32 → FP16/BF16 → INT8 (bnb) → GPTQ INT4 → AWQ INT4 → NF4 → GGUF Q2_K
         
        ← More Memory / Slower          Faster / Less Memory →
```

### Further Reading

- **LLM.int8() paper**: [arxiv.org/abs/2208.07339](https://arxiv.org/abs/2208.07339)
- **QLoRA / NF4 paper**: [arxiv.org/abs/2305.14314](https://arxiv.org/abs/2305.14314)
- **GPTQ paper**: [arxiv.org/abs/2210.17323](https://arxiv.org/abs/2210.17323)
- **AWQ paper**: [arxiv.org/abs/2306.00978](https://arxiv.org/abs/2306.00978)
- **llama.cpp / GGUF**: [github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)
- **Hugging Face Quantization Docs**: [huggingface.co/docs/transformers/quantization](https://huggingface.co/docs/transformers/main/en/quantization/overview)